<a href="https://colab.research.google.com/github/v0idengineer/MANAJEMEN-IGD/blob/main/notebooks/tf_idf_vectorizer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install pandas scikit-learn


In [ ]:
# Fungsi untuk memeriksa apakah aspek tertentu ada dalam array aspek
def detect_aspect_in_array(aspects_array, aspect):
    return 1 if aspect in aspects_array else 0

# Terapkan fungsi untuk setiap aspek pada kolom 'aspects'
for aspect in ['gameplay', 'performa', 'visualisasi', 'pemain']:
    df[aspect] = df['aspects'].apply(lambda x: detect_aspect_in_array(x, aspect))

# Periksa beberapa data untuk memastikan kolom aspek telah ditambahkan dengan benar
print(df[['lemmatized_reviews', 'aspects', 'gameplay', 'performa', 'visualisasi', 'pemain']].head())


                                  lemmatized_reviews  \
0          gim buat motivasi jadi menteri komunikasi   
1  good game rip aim everytime see enemy either I...   
2                valve support ur new game tf update   
3                                       many cheater   
4                       performa game kurang optimal   

                                             aspects  gameplay  performa  \
0  {'gameplay': 0, 'performa': 0, 'visualisasi': ...         1         1   
1  {'gameplay': 0, 'performa': 0, 'visualisasi': ...         1         1   
2  {'gameplay': 0, 'performa': 0, 'visualisasi': ...         1         1   
3  {'gameplay': 0, 'performa': 0, 'visualisasi': ...         1         1   
4  {'gameplay': 0, 'performa': 1, 'visualisasi': ...         1         1   

   visualisasi  pemain  
0            1       1  
1            1       1  
2            1       1  
3            1       1  
4            1       1  


In [ ]:
from sklearn.model_selection import train_test_split
import pandas as pd

# Fungsi untuk membagi data dengan mempertimbangkan jumlah data minimum
def split_data_by_aspect(df, aspect, train_ratio=0.8):
    # Pastikan kolom 'aspects' berisi dictionary
    def get_aspect_value(aspects, aspect):
        # Pastikan aspek adalah dictionary, jika tidak, coba parse string menjadi dictionary
        if isinstance(aspects, str):
            try:
                aspects = eval(aspects)  # Convert string to dictionary
            except:
                return 0  # Jika gagal parse, anggap tidak ada data
        return aspects.get(aspect, 0) if isinstance(aspects, dict) else 0

    # Mengambil data yang membahas aspek yang ditentukan
    aspect_data = df[df['aspects'].apply(lambda x: get_aspect_value(x, aspect) == 1)]  # Data yang membahas aspek
    other_data = df[df['aspects'].apply(lambda x: get_aspect_value(x, aspect) == 0)]   # Data yang tidak membahas aspek

    # Pastikan ada cukup data untuk pembagian
    if len(aspect_data) < 2 or len(other_data) < 2:
        raise ValueError(f"Jumlah data untuk aspek {aspect} terlalu sedikit untuk pembagian train/test.")

    # Tentukan ukuran data untuk training dan testing
    train_size_aspect = int(len(aspect_data) * train_ratio)
    test_size_aspect = len(aspect_data) - train_size_aspect

    train_size_other = int(len(other_data) * train_ratio)
    test_size_other = len(other_data) - train_size_other

    # Jika ukuran data test terlalu kecil, sesuaikan dengan ukuran yang memungkinkan
    if test_size_aspect <= 0 or test_size_other <= 0:
        # Menangani kasus di mana pembagian tidak dapat dilakukan dengan benar
        print(f"Aspek {aspect} memiliki jumlah data yang sangat terbatas, menggunakan data yang ada untuk training.")
        aspect_train, aspect_test = aspect_data, aspect_data
        other_train, other_test = other_data, other_data
    else:
        # Pembagian data jika ada cukup data untuk testing
        aspect_train, aspect_test = train_test_split(aspect_data, train_size=train_size_aspect, test_size=test_size_aspect, random_state=42)
        other_train, other_test = train_test_split(other_data, train_size=train_size_other, test_size=test_size_other, random_state=42)

    # Gabungkan data training dan testing
    train_data = pd.concat([aspect_train, other_train])
    test_data = pd.concat([aspect_test, other_test])

    return train_data, test_data

# Daftar aspek
aspects = ['gameplay', 'performa', 'visualisasi', 'pemain']

# Filter data untuk hanya bahasa Inggris (en)
df_en = df[df['language'] == 'en']

# Split data untuk setiap aspek
train_data = {}
test_data = {}

for aspect in aspects:
    try:
        train_data[aspect], test_data[aspect] = split_data_by_aspect(df_en, aspect)
    except ValueError as e:
        print(e)

# Cek jumlah data yang terbagi untuk setiap aspek
for aspect in aspects:
    if aspect in train_data:
        print(f"{aspect}: Training data size = {len(train_data[aspect])}, Test data size = {len(test_data[aspect])}")
    else:
        print(f"{aspect}: Tidak ada cukup data untuk pembagian.")


gameplay: Training data size = 7804, Test data size = 1952
performa: Training data size = 8361, Test data size = 2091
visualisasi: Training data size = 8395, Test data size = 2100
pemain: Training data size = 8113, Test data size = 2030


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import classification_report, accuracy_score, f1_score, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
import ast  # Untuk mengubah string menjadi dictionary

# Membaca dataset yang sudah dibersihkan
df = pd.read_csv("dataset_cleaned_with_aspects.csv")

# Fungsi untuk mengecek apakah minimal satu aspek dibahas
def has_at_least_one_aspect(aspects):
    aspects_dict = ast.literal_eval(aspects)  # Mengubah string menjadi dictionary
    return any(value == 1 for value in aspects_dict.values())  # Memeriksa apakah ada nilai 1 dalam dictionary

# Filter data yang membahas minimal satu aspek
df_filtered = df[df['aspects'].apply(has_at_least_one_aspect)]

# Pembagian data menjadi train dan test
train_data, test_data = train_test_split(df_filtered, test_size=0.2, random_state=42)

# TF-IDF Vectorizer
vectorizer = TfidfVectorizer(stop_words='english', max_features=5000)

# Fungsi untuk melatih model dan menghasilkan evaluasi
def train_and_evaluate(train_data, test_data, aspect):
    # Melatih model SVM
    X_train = vectorizer.fit_transform(train_data['lemmatized_reviews'])
    X_test = vectorizer.transform(test_data['lemmatized_reviews'])

    # Target variable
    y_train = train_data[aspect]
    y_test = test_data[aspect]

    model = SVC(kernel='linear', random_state=42)
    model.fit(X_train, y_train)

    # Prediksi dan evaluasi
    y_pred = model.predict(X_test)

    # Pastikan jumlah sampel antara y_test dan y_pred sama
    if len(y_test) != len(y_pred):
        raise ValueError(f"Jumlah sampel tidak konsisten: y_test={len(y_test)}, y_pred={len(y_pred)}")

    # Akurasi
    accuracy = accuracy_score(y_test, y_pred)

    # F1-score (multiclass, average='macro', 'micro', or 'weighted')
    f1 = f1_score(y_test, y_pred, average='macro')  # 'micro', 'macro', or 'weighted'

    # Confusion Matrix
    cm = confusion_matrix(y_test, y_pred)

    # Tentukan kelas yang unik di y_test dan y_pred
    unique_classes = sorted(set(y_test) | set(y_pred))

    # Pastikan hanya kelas yang valid
    if len(unique_classes) > 3:
        raise ValueError(f"Lebih dari 3 kelas ditemukan: {unique_classes}. Periksa data.")

    # Tentukan class labels menjadi 'Positif', 'Netral', 'Negatif'
    class_labels = ['Negatif', 'Netral', 'Positif']  # Sesuaikan urutan sesuai kebutuhan

    # Mengubah confusion matrix menjadi DataFrame dengan kelas yang ditemukan
    cm_df = pd.DataFrame(cm, index=class_labels[:len(unique_classes)], columns=class_labels[:len(unique_classes)])

    return f1, accuracy, cm_df

# Menyimpan hasil
results = []

# Aspects yang akan dianalisis (diambil dari keys dictionary dalam kolom aspects)
aspects_to_analyze = ['gameplay', 'performa', 'visualisasi', 'pemain']

# Visualisasi dan evaluasi untuk setiap aspek
for aspect in aspects_to_analyze:
    # Membuat kolom biner untuk setiap aspek
    train_data[aspect] = train_data['aspects'].apply(lambda x: ast.literal_eval(x).get(aspect, 0))
    test_data[aspect] = test_data['aspects'].apply(lambda x: ast.literal_eval(x).get(aspect, 0))

    # Melatih dan mengevaluasi model
    f1, accuracy, cm_df = train_and_evaluate(train_data, test_data, aspect)

    # Menyimpan hasil untuk tabel
    results.append({
        'Aspek': aspect,
        'F1-score (Macro)': f1,
        'Akurasi': accuracy
    })

    # Menampilkan confusion matrix sebagai tabel
    print(f"Confusion Matrix untuk Aspek: {aspect}")
    print(cm_df)
    print("\n" + "="*50 + "\n")

# Menyusun hasil ke dalam DataFrame
results_df = pd.DataFrame(results)

# Menampilkan hasil
print("Hasil Evaluasi Model:")
print(results_df)


ValueError: Lebih dari 3 kelas ditemukan: [0, 1, 2, 3, 4, 5, 6]. Periksa data.